In [ ]:
# !pip install pydicom
# !pip install pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg
!pip install --upgrade bitsandbytes
!pip install --upgrade transformers accelerate

In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv')

# Display the first few rows to verify
df.head(100)

In [1]:
"""
Knee abnormality report labeler — small model, batched, multi-GPU.

Same architecture as the original Qwen script (prompt -> JSON -> parse), but:
  * Qwen2.5-1.5B-Instruct instead of 7B  (~3GB fp16, fits trivially on one T4)
  * fp16, no quantization  (nf4 is SLOWER than fp16 on Turing; T4 has no bf16)
  * batched generation      (the actual 15-25x speedup)
  * chat template applied   (Instruct models degrade badly without it)
  * one full model replica per GPU, not device_map="auto" layer sharding

Kaggle 2x T4.
"""

import json
import logging
import os
import re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import pydicom
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # step up to -3B- if parse quality is poor
COMP_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
BATCH_SIZE = 32          # drop to 16 if you see OOM with long reports
MAX_INPUT_TOKENS = 1536
MAX_NEW_TOKENS = 96      # a 12-key JSON is ~70 tokens; 256 was wasting decode steps
CHECKPOINT_EVERY = 5     # batches

LABEL_COLUMNS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

SYSTEM_PROMPT = (
    "You are a musculoskeletal radiologist extracting structured findings from "
    "knee MRI reports. You reply with a single JSON object and nothing else."
)


# --------------------------------------------------------------------------- #
# DICOM metadata (unchanged behaviour, trimmed fields)
# --------------------------------------------------------------------------- #

class DicomMeta:
    KEEP = ("Modality", "SeriesDescription", "BodyPartExamined", "Laterality")

    def __init__(self, base_path):
        self.base_path = base_path
        self.cache = {}

    def get(self, study_uid):
        if study_uid in self.cache:
            return self.cache[study_uid]

        study_path = Path(self.base_path) / str(study_uid)
        if not study_path.exists():
            self.cache[study_uid] = {}
            return {}

        dcm_files = list(study_path.rglob("*.dcm"))
        if not dcm_files:
            self.cache[study_uid] = {}
            return {}

        try:
            ds = pydicom.dcmread(dcm_files[0], stop_before_pixels=True)
            meta = {}
            for key in self.KEEP:
                val = ds.get(key, "")
                val = "" if val is None else str(val).strip()
                if val and val.upper() not in ("N/A", "NONE", "NULL"):
                    meta[key] = val
            self.cache[study_uid] = meta
            return meta
        except Exception as e:  # noqa: BLE001
            logging.warning(f"Error reading DICOM for {study_uid}: {e}")
            self.cache[study_uid] = {}
            return {}


# --------------------------------------------------------------------------- #
# One model replica pinned to one GPU
# --------------------------------------------------------------------------- #

class Replica:
    def __init__(self, model_name, device):
        self.device = device
        logging.info(f"Loading {model_name} onto {device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
        ).to(device)
        self.model.eval()
        self.model.generation_config.pad_token_id = self.tokenizer.pad_token_id
        logging.info(f"Ready on {device}")

    @torch.inference_mode()
    def generate(self, prompts):
        texts = [
            self.tokenizer.apply_chat_template(
                [{"role": "system", "content": SYSTEM_PROMPT},
                 {"role": "user", "content": p}],
                tokenize=False,
                add_generation_prompt=True,
            )
            for p in prompts
        ]
        enc = self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
        ).to(self.device)

        out = self.model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=self.tokenizer.pad_token_id,
        )
        gen = out[:, enc["input_ids"].shape[1]:]
        return self.tokenizer.batch_decode(gen, skip_special_tokens=True)


# --------------------------------------------------------------------------- #
# Labeler
# --------------------------------------------------------------------------- #

class KneeAbnormalityLabeler:
    def __init__(self, model_name=MODEL_NAME,
                 checkpoint_path="labeling_checkpoint.json",
                 dicom_base_path=f"{COMP_ROOT}/train_series/",
                 n_gpus=None):
        self.checkpoint_path = checkpoint_path
        self.dicom = DicomMeta(dicom_base_path)
        self.results = {}

        if os.path.exists(checkpoint_path):
            with open(checkpoint_path) as f:
                self.results = json.load(f)
            logging.info(f"Loaded checkpoint with {len(self.results)} completed rows")

        if n_gpus is None:
            n_gpus = max(1, torch.cuda.device_count())
        devices = [f"cuda:{i}" for i in range(n_gpus)] if torch.cuda.is_available() else ["cpu"]
        self.replicas = [Replica(model_name, d) for d in devices]
        self.pool = ThreadPoolExecutor(max_workers=len(self.replicas))
        logging.info(f"{len(self.replicas)} replica(s) live")

    # ---- prompt ---------------------------------------------------------- #

    def create_prompt(self, report_text, dicom_metadata=None):
        keys = ", ".join(f'"{c}"' for c in LABEL_COLUMNS)
        prompt = (
            "Read the knee MRI report below and decide, for each of 12 conditions, "
            "whether it is present.\n\n"
            f"Output a JSON object with exactly these keys: {keys}\n\n"
            "Values:\n"
            "  1  = condition is affirmatively described\n"
            "  0  = condition is explicitly negated or described as normal/intact\n"
            '  "?" = not mentioned, or the report is equivocal\n\n'
            "Do not infer a condition from an adjacent one. A meniscal tear does not "
            "imply an ACL tear. Respect laterality: medial findings go to medial keys "
            "only.\n\n"
        )
        if dicom_metadata:
            prompt += "Acquisition details:\n"
            for k, v in dicom_metadata.items():
                prompt += f"- {k}: {v}\n"
            prompt += "\n"
        prompt += f"REPORT:\n{report_text}\n\nJSON:"
        return prompt

    # ---- parsing (kept from original, slightly tightened) ---------------- #

    @staticmethod
    def parse_response(response_text):
        if not response_text:
            return None

        cleaned = re.sub(r"```(?:json)?\s*", "", response_text, flags=re.IGNORECASE)
        cleaned = re.sub(r"```\s*$", "", cleaned)

        start = cleaned.find("{")
        if start == -1:
            return None
        end = cleaned.rfind("}")

        if end != -1 and start < end:
            try:
                return json.loads(cleaned[start:end + 1])
            except json.JSONDecodeError:
                pass

        # Truncated output: close the object and drop any trailing partial pair.
        tail = cleaned[start:]
        tail = re.sub(r",\s*\"[^\"]*\"?\s*:?\s*[^,}]*$", "", tail)
        try:
            return json.loads(tail + "}")
        except json.JSONDecodeError:
            pass

        match = re.search(r"\{[^{}]*\}", cleaned)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
        return None

    # ---- batch dispatch --------------------------------------------------- #

    def _run_batch(self, prompts):
        """Split one batch across replicas, run in parallel, reassemble in order."""
        n = len(self.replicas)
        if n == 1:
            return self.replicas[0].generate(prompts)

        chunks = [prompts[i::n] for i in range(n)]
        futures = [
            self.pool.submit(rep.generate, chunk)
            for rep, chunk in zip(self.replicas, chunks) if chunk
        ]
        outs = [f.result() for f in futures]

        merged = [None] * len(prompts)
        for i, out in enumerate(outs):
            merged[i::n] = out
        return merged

    def save_checkpoint(self):
        with open(self.checkpoint_path, "w") as f:
            json.dump(self.results, f)
        logging.info(f"Checkpoint saved: {len(self.results)} rows")

    # ---- main loop -------------------------------------------------------- #

    def label_dataframe(self, df):
        result_df = df.copy()
        if "inferred" not in result_df.columns:
            result_df["inferred"] = ""

        todo = []
        for idx in range(len(df)):
            if str(idx) in self.results:
                continue
            row = df.iloc[idx]
            if not any(pd.isna(row[c]) for c in LABEL_COLUMNS):
                continue
            if pd.isna(row["Report"]) or not str(row["Report"]).strip():
                self.results[str(idx)] = {"error": "empty_report"}
                continue
            todo.append(idx)

        logging.info(f"{len(todo)} rows need labeling")

        n_batches = (len(todo) + BATCH_SIZE - 1) // BATCH_SIZE
        parse_failures = 0

        for b in tqdm(range(n_batches), desc="Labeling"):
            batch_idx = todo[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
            prompts = []
            for idx in batch_idx:
                row = df.iloc[idx]
                meta = self.dicom.get(row["StudyInstanceUID"])
                prompts.append(self.create_prompt(str(row["Report"]), meta))

            responses = self._run_batch(prompts)

            for idx, resp in zip(batch_idx, responses):
                labels = self.parse_response(resp)
                if labels is None:
                    parse_failures += 1
                    self.results[str(idx)] = {"error": "parse_failed", "raw": resp[:200]}
                    continue

                inferred_cols = []
                for col in LABEL_COLUMNS:
                    if col not in labels:
                        continue
                    val = labels[col]
                    if val == "?" or val is None:
                        continue
                    try:
                        num = float(val)
                    except (ValueError, TypeError):
                        continue
                    if num in (0.0, 1.0):
                        result_df.at[idx, col] = int(num)
                        inferred_cols.append(col)

                if inferred_cols:
                    result_df.at[idx, "inferred"] = ",".join(inferred_cols)
                self.results[str(idx)] = {"completed": True, "inferred": inferred_cols}

            if (b + 1) % CHECKPOINT_EVERY == 0:
                self.save_checkpoint()

        self.save_checkpoint()
        logging.info(f"Parse failures: {parse_failures} / {len(todo)}")
        return result_df


def main():
    df = pd.read_csv(f"{COMP_ROOT}/train.csv")
    logging.info(f"Loaded {len(df)} rows; columns: {df.columns.tolist()}")

    labeler = KneeAbnormalityLabeler()
    labeled_df = labeler.label_dataframe(df)
    labeled_df.to_csv("train_labeled.csv", index=False)

    print("\n" + "=" * 50)
    print("LABELING SUMMARY")
    print("=" * 50)
    print(f"Total rows: {len(labeled_df)}")
    print(f"Rows with inferred labels: {labeled_df['inferred'].str.len().gt(0).sum()}")
    print("\nLabel distribution:")
    for col in LABEL_COLUMNS:
        non_nan = labeled_df[col].notna().sum()
        inferred = labeled_df["inferred"].str.contains(re.escape(col), na=False).sum()
        print(f"  {col:20s}: {non_nan:6d} total, {inferred:6d} inferred")


if __name__ == "__main__":
    main()

2026-08-29 15:28:18,451 - INFO - Loaded 4407 rows; columns: ['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
2026-08-29 15:28:18,755 - INFO - Loading Qwen/Qwen2.5-1.5B-Instruct onto cuda:0...
2026-08-29 15:28:18,920 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-29 15:28:18,921 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-29 15:28:18,939 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
2026-08-29 15:28:18,956 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

2026-08-29 15:28:19,039 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-29 15:28:19,055 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-29 15:28:19,072 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

2026-08-29 15:28:19,154 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-29 15:28:19,222 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-08-29 15:28:19,289 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
2026-08-29 15:28:19,307 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/vocab.json "HTTP/1.1 200 OK"
2026-08-29 15:28:19,325 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/vocab.json "HTTP/1.1 200 OK"


...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

2026-08-29 15:28:52,636 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-29 15:28:52,652 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/generation_config.json "HTTP/1.1 200 OK"
2026-08-29 15:28:53,611 - INFO - Ready on cuda:1
2026-08-29 15:28:53,612 - INFO - 2 replica(s) live
2026-08-29 15:28:53,821 - INFO - 4349 rows need labeling


Labeling:   0%|          | 0/136 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2026-08-29 15:30:15,753 - INFO - Checkpoint saved: 160 rows
2026-08-29 15:31:44,561 - INFO - Checkpoint saved: 320 rows
2026-08-29 15:33:06,772 - INFO - Checkpoint saved: 480 rows
2026-08-29 15:34:22,344 - INFO - Checkpoint saved: 640 rows
2026-08-29 15:35:49,044 - INFO - Checkpoint saved: 800 rows
2026-08-29 15:37:04,197 - INFO - Checkpoint saved: 960 rows
2026-08-29 15:38:28,822 - INFO - Checkpoint saved: 1120 rows
2026-08-29 15:39:53,015 - INFO - Checkpoint saved: 1280 rows
2026-08-29 15:41:17,023 - INFO - Checkpoint saved: 1440 rows
2026-08-29 15:42:44,104 - INFO - Checkpoint saved: 1600 rows
2026-08-29 15:44:06,275 - INFO - Checkpoint saved: 1760 rows
2026-08-29 15:45:29,027 - INFO 


LABELING SUMMARY
Total rows: 4407
Rows with inferred labels: 3995

Label distribution:
  ACL                 :   2775 total,   2717 inferred
  MCL                 :   2795 total,   2737 inferred
  Medial Meniscus     :   3682 total,   3624 inferred
  Lateral Meniscus    :   3507 total,   3449 inferred
  Medial OA           :   2904 total,   2846 inferred
  Lateral OA          :   2898 total,   2840 inferred
  PF OA               :   2778 total,   2720 inferred
  Effusion            :   3671 total,   3613 inferred
  Synovitis           :   2627 total,   2569 inferred
  Baker's             :   3000 total,   2942 inferred
  Contusion           :    180 total,    122 inferred
  Fracture            :     58 total,      0 inferred
